# 탐 이름 규칙: 킥 울림 빼기 후보 채점 — GPU 서버 실행 (S15P31A307-164)

ENST 드러머 1처럼 킥을 높게 조율한 세트에서 킥 울림이 탐 음높이에 섞여 탐 이름이 밀리는 문제를 고치는 규칙 후보를 채점한다. 후보는 `docs/chart-rules/tom-kick-pitch-2026-11-01/candidates.yaml`. GPU는 안 쓰고 저장해 둔 특징으로 CPU에서 채점만 한다.

- 기본 가중치 특징 `data/rule_features`: ENST·MDB·GMD·STAR·E-GMD 다섯 데이터
- 드러머 1 연습 연주 학습 특징 `data/rule_features_rc_tom_6cls_real_d1_practice`: ENST·MDB·GMD·STAR. 탐 위치를 거의 맞히는 모델이라 이름 규칙 효과가 잘 보임

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에, 이 파일을 `ssaplash/notebooks/`에 올린다.

**순서.** 1 설정 → 2 코드 덮어 풀기 → 3 채점 시작(백그라운드, 한 번만) → 4 진행 확인(여러 번) → 5 결과 묶기. 3번은 백그라운드라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번을 치고 4번으로 간다.

In [ ]:
# 1. 설정 (CPU만 씀)
import os
import sys

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
DS = "configs/bench/datasets"
CAND = "docs/chart-rules/tom-kick-pitch-2026-11-01/candidates.yaml"
# 채점 이름 → (특징 폴더, 데이터셋)
JOBS = {
    "base": ("data/rule_features", f"{DS}/enst_minus_one.yaml {DS}/mdb_drums.yaml {DS}/gmd_test.yaml {DS}/star_drums_test.yaml {DS}/egmd_test_acoustic.yaml"),
    "practice": ("data/rule_features_rc_tom_6cls_real_d1_practice", f"{DS}/enst_minus_one.yaml {DS}/mdb_drums.yaml {DS}/gmd_test.yaml {DS}/star_drums_test.yaml"),
}
OUT = {j: f"data/results/rule-eval-kick-pitch-{j}" for j in JOBS}
LOG = {j: f"{AI}/rule_eval_kick_pitch_{j}.log" for j in JOBS}
os.makedirs(RESULTS, exist_ok=True)
%cd $AI
print("CPU 코어", os.cpu_count())
!ls -d data/rule_features data/rule_features_rc_tom_6cls_real_d1_practice
!free -g | head -2
!ls -la $UPLOAD/S15P31A307_code.zip
# 확인: 폴더 2줄, 메모리 표, 코드 zip 날짜가 오늘 올린 것. 'No such file'이면 그 줄을 적어 둠

In [ ]:
# 2. 코드 덮어 풀기. 기존 폴더를 지우지 않고 위에 덮어씀 (.venv·데이터 링크·특징 파일이 그대로 남음)
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!grep -c "tom_kick_cluster_st" src/ssaplash_ai/bench/transcribers/adtof_drumsep_transcriber.py
!grep -c "^- name" $CAND
# 확인: 첫 숫자는 0이 아님(새 코드), 둘째 숫자는 7(후보 7개). 첫 숫자가 0이면 zip이 예전 것이니 로컬에서 다시 만들어 올림

In [ ]:
# 3. 채점 시작 (CPU, 백그라운드, nice 19, 한 번만). 기본 가중치 다섯 데이터는 E-GMD 1,170곡 때문에 1~2시간, 연습 연주 학습은 20~40분
#    두 작업이 각자 특징을 메모리에 올림. 1번 셀 free의 available이 100GB 아래면 practice만 먼저 띄우고 끝난 뒤 base를 띄움
import subprocess

for j, (feats, ds) in JOBS.items():
    if os.path.exists(LOG[j]):
        print(j, "이미 시작했음, 건너뜀 (다시 돌리려면 로그를 지우고 침)")
        continue
    cmd = ["nice", "-n", "19", ".venv/bin/ssaplash-bench", "rule-eval", "--features", feats, "--dataset", *ds.split(),
           "--candidates", CAND, "--out", OUT[j], "--two-tom-upper", "MID_TOM", "--mlflow-uri", "none"]
    proc = subprocess.Popen(["nohup", *cmd], cwd=AI, stdout=open(LOG[j], "w"), stderr=subprocess.STDOUT,
                            env=dict(os.environ), start_new_session=True)
    print("시작.", j, "pid", proc.pid, "로그", LOG[j])
# 확인: '시작. base pid …', '시작. practice pid …' 두 줄

In [ ]:
# 4. 진행 확인 (여러 번)
for j in JOBS:
    text = open(LOG[j], encoding="utf-8", errors="replace").read() if os.path.exists(LOG[j]) else ""
    done = "[rule-eval] 결과 →" in text
    print(f"== {j}: {'끝남' if done else '도는 중'}, 채점 줄 {text.count('f1Tom=')}개")
    if "Traceback" in text:
        print(*text.strip().splitlines()[-5:], sep="\n")
    for line in text.splitlines():
        if "f1Tom=" in line and "enst_minus_one" in line:
            print("  ", line.split(" f1Cymbal")[0])
# 확인: 끝나면 base는 채점 줄 35개(후보 7 × 데이터 5), practice는 28개(7 × 4). ENST 줄의 f1Tom이 후보마다 보임.
#   'Traceback'이 나오면 그 다섯 줄을 붙여 줌

In [ ]:
# 5. 결과 묶기. results 폴더의 tom_kick_pitch_results.zip 을 내려받아 로컬 AI/data/results/ 에 둠 (푸는 것은 세션에서)
import zipfile


def pack(zip_path, paths):
    # 폴더는 안의 파일을 전부, 파일은 그대로. 없는 경로 목록을 돌려줌
    missing = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            if os.path.isdir(p):
                for root, _, files in os.walk(p):
                    for f in files:
                        z.write(os.path.join(root, f))
            elif os.path.isfile(p):
                z.write(p)
            else:
                missing.append(p)
    return missing


missing = pack(f"{RESULTS}/tom_kick_pitch_results.zip", [*OUT.values(), *(os.path.basename(p) for p in LOG.values())])
!ls -la $RESULTS/tom_kick_pitch_results.zip
print("zip에 빠진 것:", missing or "없음")
# 확인: 'zip에 빠진 것: 없음'과 수 MB짜리 zip 한 줄